# UMAP (Uniform Manifold Approximation and Projection)
UMAP (Uniform Manifold Approximation and Projection) is a nonlinear manifold-learning technique that does a much better job. It works in two main steps:

It builds a fuzzy topological graph of the high-dimensional data by connecting each point to its nearest neighbors with a smooth distance kernel (so close points have strong connections, far points have weak or none).
It then optimizes a low-dimensional layout (usually 2D) that minimizes the cross-entropy between the high-dimensional graph and the new 2D graph, preserving both local neighborhoods and the overall global structure.
This produces cleaner, more separated clusters and more interpretable visualizations of CLIP embeddings than PCA ever could.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap

# ================================================================
# 1. PREPARE AND NORMALIZE DATA
# ================================================================
print("🔹 Step 1: Normalizing features to unit vectors (required for CLIP)...")

# CLIP uses cosine similarity → we must make every vector length = 1
img_norm = np_image_features / np.linalg.norm(np_image_features, axis=1, keepdims=True)
text_norm = np_text_features / np.linalg.norm(np_text_features, axis=1, keepdims=True)

# ================================================================
# 2. CLOSE THE "MODALITY GAP" (very important trick!)
# ================================================================
print("🔹 Step 2: Centering both modalities to remove the modality gap...")

# Image embeddings and text embeddings often live in slightly different parts of the space.
# Subtracting the mean of each group moves both clouds to the same center.
# This makes the visualization much cleaner and more accurate.
img_centered = img_norm - img_norm.mean(axis=0)
text_centered = text_norm - text_norm.mean(axis=0)

# Combine everything for UMAP (images on top, texts on bottom)
all_features = np.vstack((img_centered, text_centered))
n_samples = len(np_image_features)   # how many image-text pairs we have

print(f"   → {n_samples} image embeddings + {n_samples} text embeddings = {all_features.shape[0]} total points")

# ================================================================
# 3. APPLY UMAP (2D projection)
# ================================================================
print("🔹 Step 3: Running UMAP to project everything into 2D...")

# Easy-to-understand parameter explanations:
reducer = umap.UMAP(
    n_neighbors=2,      # Small number = focus on local clusters (good for CLIP)
    min_dist=0.2,       # How spread out points can be (0.3 gives nice separation)
    metric='cosine',    # The correct distance for CLIP embeddings
    random_state=42,    # Makes results exactly the same every time you run it
)

features_2d = reducer.fit_transform(all_features)

# Split the 2D points back into images and texts
img_2d = features_2d[:n_samples]
text_2d = features_2d[n_samples:]

print("   ✅ UMAP finished! Ready to visualize.")

# ================================================================
# 4. CREATE THE BEAUTIFUL VISUALIZATION
# ================================================================
print("🔹 Step 4: Creating the plot...")

plt.figure(figsize=(14, 10))  # nice big canvas

# Plot IMAGE points (blue circles)
plt.scatter(img_2d[:, 0], img_2d[:, 1],
            c='royalblue', marker='o', s=180,
            label='Image Embeddings',
            edgecolors='white', linewidth=2, alpha=0.9, zorder=3)

# Plot TEXT points (red triangles)
plt.scatter(text_2d[:, 0], text_2d[:, 1],
            c='crimson', marker='^', s=180,
            label='Text Embeddings',
            edgecolors='white', linewidth=2, alpha=0.9, zorder=3)

# Draw connecting lines + labels for every pair
for i in range(n_samples):
    # Semantic bridge line (shows how well image and text are aligned)
    plt.plot([img_2d[i, 0], text_2d[i, 0]],
             [img_2d[i, 1], text_2d[i, 1]],
             color='gray', linestyle='--', alpha=0.35, linewidth=1.2, zorder=1)
    
    # Label (truncated so it doesn't get too long and messy)
    caption = texts[i]
    if len(caption) > 45:
        caption = caption[:42] + "..."
    plt.text(img_2d[i, 0] + 0.06, img_2d[i, 1] + 0.06,
             caption,
             fontsize=10, fontweight='bold', color='darkslategray',
             alpha=0.95, zorder=4)

# Final plot styling
plt.title("UMAP Projection of CLIP Shared Embedding Space\n"
          "Blue = Images • Red = Texts • Dashed lines = Semantic Bridges",
          fontsize=18, pad=25, fontweight='bold')

plt.xlabel("UMAP Dimension 1", fontsize=13)
plt.ylabel("UMAP Dimension 2", fontsize=13)

plt.legend(frameon=True, facecolor='white', edgecolor='gray', fontsize=12, loc='best')
plt.grid(True, linestyle='--', alpha=0.3)

# Keep the shape natural (important for cosine space)
plt.gca().set_aspect('equal', adjustable='datalim')

plt.tight_layout()

# Optional: save a high-quality version
# plt.savefig("clip_umap_visualization.png", dpi=300, bbox_inches='tight')

plt.show()

print("🎉 Visualization complete!")